## Adding another strategy from the library

In [2]:
import pathlib
import re

import axelrod as axl
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import dask.dataframe as dd
import sklearn
from sklearn.feature_selection import RFE, SequentialFeatureSelector, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import json

In [3]:
with open("./data/original_tournament/fortran_characteristics.json", "r") as f:
    characteristics = json.load(f)

In [4]:
number_of_translated_strategies = sum(characteristic['axelrod-python_class'] is not None for characteristic in characteristics.values())
with open("../assets/number_of_translated_strategies.tex", "w") as f:
    f.write(str(number_of_translated_strategies))

In [5]:
second_tournament_strategies = [
    name for name in characteristics.keys()
    if characteristics[name]["original_rank"] is not None
]

with open("../assets/list_of_original_tournament_players.tex", "w") as f:
    for name in second_tournament_strategies:
        dictionary = characteristics[name]
        author=dictionary["author"]
        original_rank=dictionary["original_rank"]
        f.write(f"\\item {name} - Original rank: {original_rank}. Authored by {author}\n")

In [6]:
def get_turns(filename):
    """
    Read the number of turns if included in the file name
    """
    match = re.search("[0-9]+(?=(_turns))", str(filename))
    return int(match.group(0))

def get_repetitions(filename):
    """
    Read the number of repetitions if included in the file name
    """
    match = re.search("[0-9]+(?=(_repetitions))", str(filename))
    return int(match.group(0))

def read_tournament_repetitions(files, player_names=None):
    """
    Read the scores from a collection of gz files 
    representing repetitions of tournaments.
    """
    number_of_opponents = len(player_names) - 1
    dfs = []
    for gz_path in files:
        dfs.append(pd.read_csv(str(gz_path), header=None).iloc[:,0:number_of_opponents + 1])
        
        turns = get_turns(gz_path)
        
        dfs[-1] /= turns * (number_of_opponents)  # Scale all metrics
        dfs[-1].columns = player_names
        
    df = pd.concat(dfs, ignore_index=True)
    return df

def read_payoff_matrix(files):
    arrays = []
    turns = []
    repetitions = 0
    for gz_path in files:
        repetitions += get_repetitions(gz_path)
        arrays.append(np.array(pd.read_csv(str(gz_path), header=None)))  # Read through pd to deal with float conversion
        turns.append(get_turns(str(gz_path)))
    payoff_matrix = sum(array * turn for turn, array in zip(turns, arrays)) / sum(turns)

    return payoff_matrix, repetitions

def get_indices_of_players(player_names, player_index):
    """
    Returns the indices of the players in `player_names` from `player_index`
    """
    indices = []
    for player in player_names:
        indices.append(list(player_index).index(player))
    return indices

def get_results_of_sub_tournament(
    player_names, 
    payoff_matrix, 
    player_index, 
    characteristics=characteristics,
):
    player_indices = get_indices_of_players(player_names, player_index)
    player_index_mesh = np.ix_(player_indices, player_indices)
    payoff_sub_matrix = payoff_matrix[player_index_mesh]
    mean_payoffs = np.mean(payoff_sub_matrix, axis=1)
    median_payoffs = np.median(payoff_sub_matrix, axis=1)
    df = pd.DataFrame(
        {
            "Name": player_names,
            "Mean payoff": mean_payoffs,
            "Median payoff": median_payoffs,
        }
    )
    df["Rank"] = df["Mean payoff"].rank(ascending=False)
    original_ranks = []
    for name in df["Name"]:
        try:
            original_rank = characteristics[name]['original_rank']
        except KeyError:
            original_rank = None
        original_ranks.append(original_rank)
    df["Original Rank"] = original_ranks
    return df.sort_values("Rank")

def add_superscript_to_name(string):
    if ("Evolved" in string) or ("PSO" in string):
        return string + r"\textsuperscript{\textdagger}"
    return string

In [7]:
original_tournament_data_path = pathlib.Path("./data/original_tournament/")
original_tournament_scores = read_tournament_repetitions(
                                   files=original_tournament_data_path.glob("*scores.gz"), 
                                   player_names=second_tournament_strategies)

In [8]:
full_tournament_data_path = pathlib.Path("./data/full_tournament/")
full_tournament_index = pd.read_csv(
    "./data/full_tournament/players.index",
    names=("Name",),
)
full_tournament_scores = read_tournament_repetitions(
                                   files=full_tournament_data_path.glob("*scores.gz"), 
                                   player_names=full_tournament_index["Name"],
)

In [9]:
full_tournament_player_index = pd.read_csv(
    f"{full_tournament_data_path}/players.index",
    names=("Name",),
)

fortran_player_index = pd.read_csv(
    f"{original_tournament_data_path}/players.index",
    names=("Name",),
)
full_tournament_payoff_matrix, repetitions = read_payoff_matrix(
    full_tournament_data_path.glob("*payoff_matrix.gz")
)

In [10]:
indices = get_indices_of_players(
    player_names=fortran_player_index["Name"], 
    player_index=full_tournament_player_index["Name"],
)

In [11]:
ddf = dd.read_csv("./data/extra_player/main.csv")
ddf.head()

,Name,Mean payoff,Median payoff,Rank,Original Rank,number of new strategies,tournament id,Winner
0,ALLCorALLD,2.119473,2.227812,58.0,NaN,1,1,k92r
1,AON2,2.634555,3.000000,49.0,NaN,1,2,k92r
2,Adaptive Pavlov 2006,2.745572,3.000000,17.0,NaN,1,3,k92r
3,Adaptive Pavlov 2011,2.722915,3.000000,23.0,NaN,1,4,k92r
4,Adaptive,1.932526,1.854349,61.0,NaN,1,5,k92r


In [12]:
ddf.tail()

,Name,Mean payoff,Median payoff,Rank,Original Rank,number of new strategies,tournament id,Winner
801145,Nice Meta Winner: 213 players,2.557330,3.0,49.0,NaN,4,77238874,k92r
801146,NMWE Stochastic: 67 players,2.602879,3.0,43.0,NaN,4,77238875,k92r
801147,Nice Meta Winner Ensemble: 213 players,2.575274,3.0,46.0,NaN,4,77238875,k92r
801148,NMWE Memory One: 36 players,2.566805,3.0,48.0,NaN,4,77238875,k92r
801149,Nice Meta Winner: 213 players,2.557330,3.0,50.0,NaN,4,77238875,k92r


In [13]:
single_strategy_ranking_df = ddf[ddf["number of new strategies"] == 1].sort_values("Rank").compute()
single_strategy_ranking_df = single_strategy_ranking_df[["Name", "Mean payoff", "Rank", "Winner"]]
single_strategy_ranking_df = single_strategy_ranking_df.rename(columns={"Mean payoff": "Mean Score"})
single_strategy_ranking_df["Rank"] = single_strategy_ranking_df["Rank"].astype(int)
single_strategy_ranking_df = single_strategy_ranking_df.set_index("Name")
single_strategy_ranking_df.index = single_strategy_ranking_df.index.map(add_superscript_to_name)

In [14]:
alternate_winners_proportion_df = pd.DataFrame()
for i in range(1, 4 + 1):
    alternate_winners_proportion_df = alternate_winners_proportion_df.join(
        pd.DataFrame(ddf[(ddf["number of new strategies"] == i)]["Winner"].value_counts().compute()),
        how="outer",
    )
    total = alternate_winners_proportion_df["count"].sum()
    alternate_winners_proportion_df["count"] = alternate_winners_proportion_df["count"] / total
    alternate_winners_proportion_df = alternate_winners_proportion_df.rename(columns={"count": f"{i} New (N = {int(total / i)})"})

In [15]:
alternate_winners_proportion_df.index = alternate_winners_proportion_df.index.map(add_superscript_to_name)

In [16]:
alternate_winners_proportion_df.tail(8).fillna(0).sum(axis=0)

1 New (N = 209)         1.000000
2 New (N = 21736)       0.998942
3 New (N = 1499784)     0.996743
4 New (N = 77238876)    0.982354
dtype: float64

In [34]:
table = alternate_winners_proportion_df.fillna(0).copy()
table = table[table.index.isin(characteristics.keys())]

table.loc["Sum"] = table.sum(axis=0, numeric_only=True)
table = table.round(5)

table.index = [add_superscript_to_name(i) if i != "Sum" else i for i in table.index]

with open("../assets/extra_strategies_summary.tex", "w") as f:
    f.write(table.to_latex(float_format="%.5f"))

table

,1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
k32r,0.00000,0.00000,0.00001,0.00079
k41r,0.00000,0.00000,0.00001,0.00000
k42r,0.14833,0.26941,0.36640,0.21723
k44r,0.00000,0.00023,0.00057,0.00461
k49r,0.00000,0.00014,0.00035,0.00023
k60r,0.00478,0.01118,0.01882,0.00451
k75r,0.00000,0.00051,0.00245,0.00086
k92r,0.84689,0.71747,0.60814,0.75412
Sum,1.00000,0.99894,0.99674,0.98235


In [20]:
ranked_df[ranked_df == 1].count().sort_values(ascending=False)

k92r        15827
k42r         7877
k75r          570
k60r          261
k82r          208
            ...  
k57r            0
k58r            0
k59r            0
k61r            0
krandomc        0
Length: 63, dtype: int64

In [1]:
full_table = alternate_winners_proportion_df.fillna(0).copy()

values = [no_addition.get(i, 0) for i in scores.index]
#full_table.insert(loc=0, column="0 New (N = 1)", value=values)

full_table.loc["Sum"] = table.sum(axis=0, numeric_only=True)
full_table = table.round(3)

full_table.index = [add_superscript_to_name(i) if i != "Sum" else i for i in table.index]

with open("../assets/extra_strategies_summary_full_table.tex", "w") as f:
    f.write(full_table.to_latex(float_format="%.3f"))

NameError: name 'alternate_winners_proportion_df' is not defined

In [22]:
full_table

,0 New (N = 1),1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
k32r,0.008,0.000,0.000,0.000,0.001
k41r,0.000,0.000,0.000,0.000,0.000
k42r,0.315,0.148,0.269,0.366,0.217
k44r,0.004,0.000,0.000,0.001,0.005
k49r,0.000,0.000,0.000,0.000,0.000
k60r,0.010,0.005,0.011,0.019,0.005
k75r,0.023,0.000,0.001,0.002,0.001
k92r,0.633,0.847,0.717,0.608,0.754
Sum,0.993,1.000,0.999,0.997,0.982


In [29]:
alternate_winners_proportion_df[alternate_winners_proportion_df.index.isin(ranked_df.columns)]

,1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
Winner,,,,
k32r,NaN,NaN,0.000007,7.891363e-04
k41r,NaN,NaN,0.000013,1.294685e-08
k42r,0.148325,0.269415,0.366403,2.172321e-01
k44r,NaN,0.000230,0.000568,4.608418e-03
k49r,NaN,0.000138,0.000347,2.311530e-04
k60r,0.004785,0.011180,0.018820,4.513932e-03
k75r,NaN,0.000506,0.002448,8.636713e-04
k92r,0.846890,0.717473,0.608137,7.541159e-01


In [30]:
alternate_winners_proportion_df

,1 New (N = 209),2 New (N = 21736),3 New (N = 1499784),4 New (N = 77238876)
Winner,,,,
Adaptive Tit For Tat: 0.5,NaN,0.000092,2.320334e-04,4.445818e-04
"EugineNier: (D,)",NaN,NaN,6.667627e-07,1.942027e-07
EvolvedLookerUp2_2_2\textsuperscript{\textdagger},NaN,NaN,NaN,1.294685e-08
Firm But Fair,NaN,0.000046,1.233511e-04,1.322585e-03
"First by Stein and Rapoport: 0.05: (D, D)",NaN,NaN,1.333525e-06,2.589370e-08
GTFT: 0.33,NaN,NaN,2.000288e-05,4.045631e-04
Gradual,NaN,NaN,1.333525e-06,2.200964e-06
Meta Majority Long Memory: 134 players,NaN,NaN,NaN,9.062794e-08
Meta Majority: 213 players,NaN,NaN,NaN,7.768109e-08
